<a href="https://colab.research.google.com/github/Anushka-24-DataScience/Generative-AI/blob/main/llama_chat_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install huggingface_hub
!pip install llama-cpp-python==0.1.78
!pip install numpy==1.23.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.1.78-cp312-cp312-linux_x86_64.whl size=296684 sha256=99d5077bc000132f46ebbf2e4c1572be555d5a34ee39fa7c098a7994d2b165c0
  Stored in directory: /root/.cache/pip/wheels/79/ca/85/03c32b3b07d393042bf1c9f8d0349b96190a4f1c4f2b69f2f3
Successfully built llama-cpp-python
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 74.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting require


1.define which model file to use

2.download it from the Hugging Face Hub

3.instantiate llama_cpp.Llama with resource settings

4.build a prompt/template

5.call the model with generation settings

6.inspect the response






1) Model identity

model_name_or_path: the HF repo identifier (or local path) that contains the model files.

model_basename: the exact filename in that repo to download/use.it’s a GGML .bin (quantized) file — optimized for llama.cpp-style runtimes (smaller, lower VRAM).

In [3]:
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGML"
model_basename = "llama-2-13b-chat.ggmlv3.q5_1.bin"  # the model is in .bin format


2) Downloading the model file from Hugging Face
from huggingface_hub import hf_hub_download

hf_hub_download(...) downloads the requested file and returns the local path to the cached file.

The file is cached (typically under ~/.cache/huggingface/hub/...) so repeated runs won’t re-download it.

model_path is what you pass to the llama runtime to load the model.

In [4]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

In [5]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

model_path


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


llama-2-13b-chat.ggmlv3.q5_1.bin:   0%|          | 0.00/9.76G [00:00<?, ?B/s]

'/root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGML/snapshots/3140827b4dfcb6b562cd87ee3d7f07109b014dd0/llama-2-13b-chat.ggmlv3.q5_1.bin'

Import and instantiate the Llama runtime
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_threads=2,      # CPU cores
    n_batch=512,      # batch size for internal processing
    n_gpu_layers=32   # number of transformer layers to run on GPU
)


Llama(...) loads the model at model_path into memory and prepares the inference engine.

n_threads: number of CPU threads/cores to use for computation. More threads → faster CPU inference but uses more CPU.

n_batch: internal batch/sequence chunk size for processing. It must be between 1 and the context window n_ctx (the maximum context length of the model). Larger n_batch can yield faster throughput but requires more RAM/VRAM.

n_gpu_layers: in llama.cpp integrations this often controls how many of the model’s layers are placed on the GPU for acceleration. Increasing it uses more GPU memory but speeds up inference. If you only have CPU, keep it 0 or small. This is hardware-dependent — tune based on your GPU VRAM and the quantized model’s memory footprint.

In [7]:
llm = Llama(
    model_path=model_path,
    n_threads=2,      # CPU cores
    n_batch=512,      # Should be between 1 and n_ctx; consider VRAM
   # n_gpu_layers=32   # Change based on your model and GPU VRAM
)


AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | VSX = 0 | 


Build the prompt (user/system + instruction)
prompt: the user instruction you want the model to act on.

prompt_template: wraps the prompt in a small conversational template with a SYSTEM instruction (to influence style/behavior), the USER content, and then a place for the model (ASSISTANT:) to reply.

Templates like this are used to supply context and guardrails for the model’s reply (politeness, completeness, format etc).

In [12]:
prompt = "What is linear regression"


In [13]:
prompt_template = f"""
SYSTEM: You are a helpful, respectful and honest assistant. Always answer as helpfully as possible.

USER: {prompt}

ASSISTANT:
"""


Calling the model to generate text

This is the inference call. Here’s what each argument does and the usual tradeoffs:

prompt_template: the input text the model conditions on (system + user prompt).

max_tokens=256: maximum number of new tokens the model will generate. Shorter → less output; larger → longer answers and more compute.

temperature=0.5:

Controls randomness. 0.0 is deterministic/greedy, >0 adds stochasticity.

Lower values (0.1–0.5) produce more focused, deterministic answers; higher values (0.7–1.0) produce more diverse answers.

top_p=0.95 (nucleus sampling):

The model considers the smallest set of tokens whose cumulative probability ≥ top_p, then samples from them.

top_p + temperature are commonly used together; top_p=0.95 is a conservative default giving safe diversity.

top_k=150:

The model restricts sampling to the top k most probable tokens. A larger k gives more choices; smaller k makes it more conservative.

You can use top_k or top_p or both — they interact.

repeat_penalty=1.2:

Penalizes repeated tokens to reduce loops or repetitive outputs. Values >1.0 discourage repetition; 1.0 means no penalty.

echo=True:

Return the prompt text together with the generated tokens. Useful for debugging or when you want the full conversation. If you just want the model’s reply, set echo=False.

In [14]:
response = llm(
    prompt_template,
    max_tokens=256,
    temperature=0.5,
    top_p=0.95,
    repeat_penalty=1.2,
    top_k=150,
    echo=True
)


Llama.generate: prefix-match hit


6) Inspecting the response

response is usually a dictionary-like object with the generated text and metadata (timings, tokens, etc.). Different wrappers/versions expose slightly different keys.

Common ways to see what you got:

print(response)  


Then examine the keys; typical shapes:

response['choices'][0]['text'] or

response['choices'][0]['message']['content']
or sometimes response['text'] depending on the wrapper version. If unsure, print response.keys() or repr(response) and inspect.

In [15]:
print(response)


{'id': 'cmpl-5932fa65-7869-4a5f-9245-fd6ec3cd1957', 'object': 'text_completion', 'created': 1764945462, 'model': '/root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGML/snapshots/3140827b4dfcb6b562cd87ee3d7f07109b014dd0/llama-2-13b-chat.ggmlv3.q5_1.bin', 'choices': [{'text': "\nSYSTEM: You are a helpful, respectful and honest assistant. Always answer as helpfully as possible.\n\nUSER: What is linear regression\n\nASSISTANT:\nHi there! Linear regression is a statistical method used to predict the value of a continuous outcome variable based on one or more predictor variables. It is a type of regression analysis that assumes a straight line relationship between the independent and dependent variables. The goal of linear regression is to create a linear equation that best predicts the value of the outcome variable based on the values of the predictor variables.\n\nIn simple terms, linear regression can be thought of as a tool for making predictions about future outcomes base